# 02 — Dataset overview

Inventory, class balance, file integrity, trial-duration distribution per
group (the duration confound check), Doppler-axis-size sanity check.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.io as sio

plt.rcParams["figure.dpi"] = 110
plt.rcParams["figure.figsize"] = (10, 4)

from tqdm.notebook import tqdm
from src.data_loader import DATA_ROOT, iter_trials


## 1. Build the inventory DataFrame

In [ ]:
rows = []
for tp in iter_trials():
    rows.append({
        "subject_id": tp.subject_id, "group": tp.group,
        "test": tp.test, "trial": tp.trial,
        "size_mb": tp.path.stat().st_size / 1e6,
        "path": str(tp.path),
    })
inv = pd.DataFrame(rows)
print(f"{len(inv)} files across {inv['subject_id'].nunique()} subjects")
inv.head()


## 2. Class balance

In [ ]:
by_group = inv.groupby("group")["subject_id"].nunique()
print(by_group)

fig, ax = plt.subplots(figsize=(5, 5))
ax.pie(by_group.values, labels=[f"{g}\n(n={n})" for g, n in by_group.items()],
       autopct="%1.0f%%", colors=["#4c8", "#e66"])
ax.set_title("Subjects by group")
plt.show()


## 3. File counts per subject

Nominal is 6 (2 tests × 3 trials).

In [ ]:
counts = inv.groupby(["subject_id", "group"]).size().reset_index(name="n_files")
anomalies = counts[counts["n_files"] != 6]
print("Subjects with non-standard file counts:")
print(anomalies)

fig, ax = plt.subplots(figsize=(14, 3))
colours = counts["group"].map({"control": "#4c8", "pd": "#e66"}).values
ax.bar(counts["subject_id"], counts["n_files"], color=colours)
ax.axhline(6, color="k", linestyle="--", alpha=0.5)
ax.set_ylabel("files")
plt.xticks(rotation=90, fontsize=7)
plt.title("Files per subject (dashed = expected 6)")
plt.tight_layout(); plt.show()


## 4. File-size distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(inv["size_mb"], bins=40, color="#888", edgecolor="white")
ax.set_xlabel("file size [MB]")
ax.set_ylabel("count")
ax.set_title(f"{len(inv)} .mat files — median {inv['size_mb'].median():.0f} MB")
plt.show()


## 5. Trial-duration distribution per group

Read just `t_axis_target` from each `.mat` (small, fast). MATLAB v5
format → `scipy.io.loadmat`.

In [ ]:
durations = []
for r in tqdm(inv.to_dict("records"), desc="reading t_axis"):
    try:
        m = sio.loadmat(r["path"], variable_names=["t_axis_target"], squeeze_me=True)
        t = np.asarray(m["t_axis_target"]).ravel()
        durations.append(float(t.max() - t.min()) if t.size > 1 else np.nan)
    except Exception:
        durations.append(np.nan)
inv["duration_s"] = durations
print(inv.groupby("group")["duration_s"].describe().round(2))


## 5b. Duration confound check

**Known issue:** PD trials in this dataset are systematically a few seconds
longer than control trials. Any feature that sums energy over time will
correlate with duration — and therefore with group — even if the gait
content is identical. This is why `src/features.py` uses means and ratios
only (and explicitly does **not** include `total_energy`-style features).


In [ ]:
ctrl = inv.loc[inv["group"] == "control", "duration_s"].dropna()
pd_ = inv.loc[inv["group"] == "pd", "duration_s"].dropna()
diff_pct = 100 * (pd_.median() - ctrl.median()) / ctrl.median()
print(f"control median: {ctrl.median():.2f} s   (n={len(ctrl)})")
print(f"PD median:      {pd_.median():.2f} s   (n={len(pd_)})")
print(f"PD is {diff_pct:+.1f}% longer than control")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(ctrl, bins=30, alpha=0.7, label="control", color="#4c8")
ax.hist(pd_,  bins=30, alpha=0.7, label="pd",      color="#e66")
ax.axvline(ctrl.median(), color="#4c8", linestyle="--")
ax.axvline(pd_.median(),  color="#e66", linestyle="--")
ax.set_xlabel("trial duration [s]")
ax.set_ylabel("count")
ax.legend()
ax.set_title("Trial-duration distribution — duration is a known confound")
plt.tight_layout(); plt.show()


## 6. Doppler-axis size distribution

If the FFT size varies trial-to-trial, this surfaces it.

In [ ]:
dop_n = []
for path in tqdm(inv["path"], desc="reading doppler_axis"):
    try:
        m = sio.loadmat(path, variable_names=["doppler_axis"], squeeze_me=True)
        dop_n.append(int(np.asarray(m["doppler_axis"]).size))
    except Exception:
        dop_n.append(-1)
inv["doppler_n"] = dop_n
print(inv["doppler_n"].value_counts().sort_index())


## 7. Save the inventory

In [ ]:
out = ROOT / "outputs" / "metrics" / "inventory.csv"
out.parent.mkdir(parents=True, exist_ok=True)
inv.to_csv(out, index=False)
print(f"saved {out}")
inv.head()


### Takeaways

- Class balance is ~57% control / ~43% PD.
- `fisp_022` (7 files) and `fisp_048` (5 files) are the two known anomalies; verify they appear in the table above.
- PD trials are systematically longer → keep duration off the feature list, and rely on `src.features` (duration-invariant by construction).
- A consistent Doppler-axis size across all files means preprocessing can use a fixed `out_size` without padding.
